In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests


# Load the integrated MEDUSA dataset
df = pd.read_csv(
    '/home/rt334/Downloads/MEDUSA_Master_ULTIMATE.csv',
    low_memory=False
)


# Methylation age-acceleration clocks used for the correlation analysis
clocks = [
    'ageAcc.Horvath',
    'ageAcc.Hannum',
    'ageAcc.Levine',
    'ageAcc.Hovarth2',
    'ageAcc.PedBE',
    'ageAcc.Wu',
    'ageAcc.TL',
    'ageAcc.BLUP',
    'ageAcc.EN'
]


# Give the clocks cleaner names for the output table
clock_names = {
    'ageAcc.Horvath': 'Horvath',
    'ageAcc.Hannum': 'Hannum',
    'ageAcc.Levine': 'Levine (PhenoAge)',
    'ageAcc.Hovarth2': 'Horvath (skin & blood)',
    'ageAcc.PedBE': 'PedBE',
    'ageAcc.Wu': 'Wu',
    'ageAcc.TL': 'Telomere Length (TL)',
    'ageAcc.BLUP': 'BLUP',
    'ageAcc.EN': 'Elastic Net (EN)'
}


# Pull out the immune-cell estimates from the three deconvolution methods
immune_cols = [
    column for column in df.columns
    if column.startswith(('ConsensusTME:', 'Cibersort:', 'xCell:'))
]

print(f'Found {len(clocks)} methylation clocks')
print(f'Found {len(immune_cols)} immune features\n')


# Convert everything used in the correlations to numeric values
for column in clocks + immune_cols:
    df[column] = pd.to_numeric(df[column], errors='coerce')


# Test every methylation clock against every immune feature using Spearman correlation
results = []

for clock in clocks:
    for immune_feature in immune_cols:

        pair_data = df[[clock, immune_feature]].dropna()

        # Ignore comparisons where too few patients have both measurements
        if len(pair_data) < 10:
            continue

        # There is nothing to correlate if either variable has no variation
        if (
            pair_data[clock].nunique() < 2
            or pair_data[immune_feature].nunique() < 2
        ):
            continue

        rho, p_value = spearmanr(
            pair_data[clock],
            pair_data[immune_feature]
        )

        # Skip any comparisons that still return missing statistics
        if np.isnan(rho) or np.isnan(p_value):
            continue

        results.append({
            'Epigenetic clock': clock_names[clock],
            'Immune feature': immune_feature,
            'n': len(pair_data),
            'Spearman rho': rho,
            'p-value': p_value
        })


results_df = pd.DataFrame(results)


# Correct for the large number of clock-by-immune-feature comparisons
if not results_df.empty:

    results_df['FDR'] = multipletests(
        results_df['p-value'],
        method='fdr_bh'
    )[1]

    results_df = results_df.sort_values(
        'p-value'
    ).reset_index(drop=True)


    # Keep full precision for the calculations and only round the version being saved
    output_df = results_df.copy()

    output_df['Spearman rho'] = output_df['Spearman rho'].round(3)
    output_df['p-value'] = output_df['p-value'].round(4)
    output_df['FDR'] = output_df['FDR'].round(4)

    output_df.to_csv(
        'table_clocks_vs_immune_full.csv',
        index=False
    )


    # Summarise how many results are nominally significant and how many survive FDR
    nominal_count = (results_df['p-value'] < 0.05).sum()
    fdr_count = (results_df['FDR'] < 0.05).sum()

    print(
        f'Tested {len(results_df)} methylation clock × immune-feature pairs'
    )

    print(
        f'Nominally significant: {nominal_count} '
        f'(p < 0.05)'
    )

    print(
        f'FDR-significant: {fdr_count} '
        f'(FDR < 0.05)'
    )

    print(
        f'Minimum FDR: {results_df["FDR"].min():.3f}\n'
    )

    # Print the strongest associations so they can be checked quickly
    print(
        output_df.head(15).to_string(index=False)
    )

    print(
        '\nsaved table_clocks_vs_immune_full.csv'
    )

else:
    print('No valid clock and immune-feature comparisons were available.')

Found 9 methylation clocks
Found 77 immune features

Tested 684 methylation clock × immune-feature pairs
Nominally significant: 53 (p < 0.05)
FDR-significant: 0 (FDR < 0.05)
Minimum FDR: 0.210

      Epigenetic clock                       Immune feature  n  Spearman rho  p-value    FDR
                  BLUP                            xCell:aDC 56        -0.464   0.0003 0.2096
               Horvath                Cibersort:T cells CD8 56         0.428   0.0010 0.2096
               Horvath  Cibersort:T cells follicular helper 56         0.423   0.0012 0.2096
                Hannum Cibersort:T cells CD4 memory resting 56        -0.413   0.0015 0.2096
               Horvath         Cibersort:NK cells activated 56         0.410   0.0017 0.2096
               Horvath Cibersort:T cells CD4 memory resting 56        -0.395   0.0026 0.2096
     Levine (PhenoAge)              Cibersort:B cells naive 56        -0.393   0.0027 0.2096
                  BLUP          xCell:MicroenvironmentScore 56

In [6]:
import pandas as pd
import matplotlib.pyplot as plt


# Check how the p-values are distributed across all of the
# methylation clock and immune feature comparisons

df = pd.read_csv('table_clocks_vs_immune_full.csv')

n_tests = len(df)
n_bins = 20

# Work out roughly how many results would fall into each bin
# if the p-values were evenly distributed
expected_per_bin = n_tests / n_bins


# Plot all of the p-values together
fig, ax = plt.subplots(figsize=(9, 6))

ax.hist(
    df['p-value'],
    bins=n_bins,
    range=(0, 1),
    color='slategrey',
    edgecolor='white',
    linewidth=0.8,
    alpha=0.9
)

# Add a reference line showing what would be expected under the null
ax.axhline(
    expected_per_bin,
    color='firebrick',
    linestyle='--',
    linewidth=1.5,
    label=f'Expected under null ({expected_per_bin:.1f} per bin)'
)


# Compare the number of small p-values with the number expected by chance
low_p_count = (df['p-value'] < 0.05).sum()
expected_low_p = n_tests * 0.05


# Tidy up the figure
ax.set_xlabel('p-value', fontsize=11)
ax.set_ylabel('Number of tests', fontsize=11)

ax.set_title(
    'Distribution of p-values for methylation clock–immune associations',
    fontsize=12,
    fontweight='bold',
    pad=12
)

ax.legend(
    loc='upper right',
    frameon=False,
    fontsize=9
)

# Remove the extra borders to keep the plot simple
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()

# Save the finished figure
plt.savefig(
    'figure_clocks_vs_immune_pvalue_histogram.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.close()

print('saved figure_clocks_vs_immune_pvalue_histogram.png')


# Print a quick check of how many results fall below p < 0.05
print(
    f'\nTests with p < 0.05: {low_p_count} observed '
    f'vs {expected_low_p:.1f} expected under a uniform null distribution'
)

# Keep in mind that some of the immune estimates are related to each other
print(
    'The tests are not fully independent because related immune-cell '
    'features are measured across CIBERSORT, ConsensusTME and xCell.'
)

saved figure_clocks_vs_immune_pvalue_histogram.png

Tests with p < 0.05: 53 observed vs 34.2 expected under a uniform null distribution
The tests are not fully independent because related immune-cell features are measured across CIBERSORT, ConsensusTME and xCell.
